# Rough Implementation of Blueprint
 This notebook contains a rough implementation of the Blueprint. It is used to:
 - **Verify** if the concept works in reality
 - **Brainstorm** different **approaches** for each solution
- Identify **edge cases** in the solution.

Here in our demo we will use the **"Qwen 3.5 9B Q4 K M" (in GGUF format)** Open Source model as the central control brain.
And to use that LLM we will use **Llama.cpp (Python Library)**.

In [3]:
from llama_cpp import Llama, LlamaGrammar
# Llama grammar is required for forcing the output to be in our expected format.

We are **simulating** the industry server on our local computer.

Since on my **Mac** model is located in the downloads folder `/Users/noyan/downloads/Qwen3.5-9B-Q4_K_M.gguf`

For real work it needs alot more context to breath, but for testing purpose I am keeping it small `2000`, so that it consumes less RAM.

In [4]:
qwen = Llama(
    model_path = "/Users/noyan/downloads/Qwen3.5-9B-Q4_K_M.gguf",
    n_ctx = 2000,
    verbose = False # For agentic task we don't need verbose responses
)
print("Model Loaded Successfully!!")

output = qwen.create_chat_completion(
    messages=[{"role": "system", "content": "Hello How can I help you?"},
              {"role": "user", "content": "Hello"}],
    temperature=0,
    max_tokens=1000,
)
print(output)

Model Loaded Successfully!!
{'id': 'chatcmpl-6ffdb351-73d0-4891-86be-e68a1cab1b8c', 'object': 'chat.completion', 'created': 1790189857, 'model': '/Users/noyan/downloads/Qwen3.5-9B-Q4_K_M.gguf', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Hello! How can I help you today? Feel free to ask me questions, need advice, or just want to chat.'}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 25, 'completion_tokens': 25, 'total_tokens': 50}}


In [5]:
completion = output["choices"][0]["message"]["content"]
print(completion)

Hello! How can I help you today? Feel free to ask me questions, need advice, or just want to chat.


For now, we first need to brainstorm on the capabilities of the agent and design the execution for each of the task.
Capabilities:
1. Read/Write Files
2. List files, can create directories
3. Fetch the metadata for each file.
4. Managing git like log and version control, where each commit is trackable.
5. Run command in the sandbox, code execution (Python or JavaScript or both)

We will store the relative path of the workspace of the agent in the `rel_path` variable.
We will use `pathlib` to deal with paths.

- `.resolve()` creates absolute path from relative path, with respect to working directory
- `Path()` converts the input into a path variable.
- `.mkdir(exist_ok = True, parents = True)` It creates directory at the given location, `exist_ok = True` ensures no error occurs if the directory already exists, `parents = True` creates any parent directory if missing instead of throwing errors.

In [6]:
from pathlib import Path
rel_path = Path('./workspace').resolve()
print(rel_path)
rel_path.mkdir(exist_ok = True, parents = True)

/Users/noyan/Desktop/Smart India Hackathon/AgenticAI/Blueprint/workspace


Since we are simulating the project here, I will create a directory called `LocalStorage`, which behaves as the local data base of the industry.

The **central_backend.py** will act as a **Backbone** for the project.

The **sandbox.py** will help in code execution by **LLM**.

The **Interpreter** directory contains interpreters for different types of files.

The **LLM** directory will contain the open-source models in gguf format.

The **cli.py** will be the Command Line Interface Application to use the agent.

The **GUI** directory will be the Graphical User Interface Application to use the Agent.

Since **Central Backend** is the _spinal cord_ of this project, which controls the entire data flow in the project, it needs to be build first.

The core algorithm behind **Central Backend** is:
1. Take **User's Input**.
2. Create a Plan with **Milestones**, each milestone with **multiple tasks**, with the help of LLM.
3. Proceed for tasks that could be done **simultaneously**.
4. **Feed** the output along with the Plan to the LLM.
5. Get the new tasks and **repeat** from step 3, until all tasks are finished.
6. Show the **final inference** of the session.

In [7]:
# Rough Implementation of the above algorithm. (fixed)
import json

context = []


def read(rel: Path) -> str:
    p = Path(rel)
    if not p.is_file():
        return f"ERROR: file not found: {rel}"
    try:
        text = p.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return "ERROR: file is not valid UTF-8 text"
    if len(text) > 1500:
        return text[:1500] + f"\n...[truncated, {len(text)} chars total]"
    return text


def list_files(rel: Path) -> str:
    p = Path(rel)
    if not p.is_dir():
        return f"ERROR: not a directory: {rel}"
    entries = sorted(p.iterdir(), key=lambda e: (e.is_file(), e.name.lower()))
    if not entries:
        return "(empty directory)"
    return "\n".join(f"{'[dir] ' if e.is_dir() else '      '}{e.name}" for e in entries)


def execute_task(step, plan, path) -> bool:
    """Ask the model for the next action, execute it, append to context.
    Returns True if the model gave a final answer (loop should stop)."""
    global context

    prompt = (
        "You are a task executer, giving out the task to be executed according "
        "to the user query and outputs of previously executed tasks, if no task "
        "has been executed then the context would be empty. These are the tasks "
        "that you can execute: Read File, List files. For Read Task, respond "
        'this valid json {"type": "task", "task_type":"read", "location":"location '
        'of file to read"}, For List files task give this valid json '
        '{"type": "task", "task_type":"list", "location":"location of folder to '
        'list its components"}, If the tasks are completed and you would like to '
        'end this session then respond with this valid json {"type": "final_answer", '
        '"task_type":"answer", "content":"Here comes what ever you would like to '
        'say..."}. Remember always respond in valid json and nothing else.'
    )

    prompt += f"\nHere is the context:\n{context}"

    task = qwen.create_chat_completion(
        messages=[
            {"role": "system", "content": prompt},
            {
                "role": "user",
                "content": f"Here is the user's query:\n{plan}\nAnd this is the Workspace location:\n{path}",
            },
        ],
        temperature=0.15,
        max_tokens=2000,
    )

    raw = task["choices"][0]["message"]["content"]

    try:
        output = json.loads(raw)
    except json.JSONDecodeError:
        print(f"ERROR: model returned invalid JSON:\n{raw}")
        # Feed the error back so the model can self-correct next turn.
        context.append({
            "type": "task",
            "task_type": "error",
            "output": "Your last response was not valid JSON. Respond with valid JSON only.",
            "location": None,
        })
        return False

    print(output)

    if output.get("type") == "task" and output.get("task_type") == "read":
        loc = Path(output["location"])
        print(f"Reading from {loc}")
        cont = read(loc)

        context.append({
            "type": "task",
            "task_type": "read",
            "output": cont,
            "location": output["location"],
        })
        print(context)

    elif output.get("type") == "task" and output.get("task_type") == "list":
        loc = Path(output["location"])
        print(f"Listing from {loc}")
        cont = list_files(loc)

        context.append({
            "type": "task",
            "task_type": "list",
            "output": cont,
            "location": output["location"],
        })
        print(context)

    elif output.get("type") == "final_answer" and output.get("task_type") == "answer":
        cont = output["content"]
        print(cont)
        return True

    else:
        print(f"ERROR: unrecognized model output shape: {output}")
        context.append({
            "type": "task",
            "task_type": "error",
            "output": "Unrecognized response shape. Use one of the documented JSON formats.",
            "location": None,
        })

    return False


def initialize(user_input, rel_path="."):
    global context
    context = []

    MAX_STEPS = 8
    step = 0

    while step <= MAX_STEPS:
        finished = execute_task(step, user_input, rel_path)
        if finished:
            break
        step += 1
    else:
        print("Reached max steps without a final answer.")


initialize("Summarize what is there in the workspace folder", rel_path=".")

{'type': 'task', 'task_type': 'list', 'location': '.'}
Listing from .
[{'type': 'task', 'task_type': 'list', 'output': '[dir] workspace\n      Blueprint of SIH26117.pdf\n      blueprint_implementation.ipynb\n      sample_qwen_output.json', 'location': '.'}]
{'type': 'task', 'task_type': 'list', 'location': '.'}
Listing from .
[{'type': 'task', 'task_type': 'list', 'output': '[dir] workspace\n      Blueprint of SIH26117.pdf\n      blueprint_implementation.ipynb\n      sample_qwen_output.json', 'location': '.'}, {'type': 'task', 'task_type': 'list', 'output': '[dir] workspace\n      Blueprint of SIH26117.pdf\n      blueprint_implementation.ipynb\n      sample_qwen_output.json', 'location': '.'}]
{'type': 'task', 'task_type': 'list', 'location': '.'}
Listing from .
[{'type': 'task', 'task_type': 'list', 'output': '[dir] workspace\n      Blueprint of SIH26117.pdf\n      blueprint_implementation.ipynb\n      sample_qwen_output.json', 'location': '.'}, {'type': 'task', 'task_type': 'list', '

KeyboardInterrupt: 